In [2]:
# Project Two Dashboard, made by Stephen DaSilva

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table, no_update
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# imports CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Instantiates CRUD module, hard-coded credentials
username = "aacuser"
password = "SNHU1234"
HOST = 'localhost' 
PORT = 27017 
DB = 'aac' 
COL = 'animals' 
shelter = AnimalShelter(username, password, HOST, PORT, DB, COL) # Instantiates object

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Loads Grazioso Salvare's Logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
html.Div(
    style={ # centers image and identifier
        'display': 'flex',
        'alignItems': 'center', 
        'justifyContent': 'center',
        'gap': '20px',
        'marginBottom': '20px'
    },
    children=[
        html.A( # Adds logo with anchor url tag
            href="https://www.snhu.edu",
            target="_blank",
            children=[
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()), # inserts image
                    style={'height': '80px'} # makes image relatively small
                )
            ]
        ),
        html.H1("Stephen DaSilva's Dashboard") # Identifier
    ]
),
    html.Hr(),
    html.Div(className='buttonRow', # Adds filtering options
            style={'display': 'flex', 'justifyContent': 'center'},
            children=[
                dcc.RadioItems(
                    id='filter-type',
                    options=[
                        {'label': 'Water Rescue', 'value': 'Water'},
                        {'label': 'Mountain Rescue', 'value': 'Mountain'},
                        {'label': 'Disaster Rescue', 'value': 'Disaster'},
                        {'label': 'Reset', 'value': 'Reset'}
                    ],
                    inline=True, # makes them appear in a row
                    inputStyle={'margin-right': '6px', 'margin-left': '12px'}
            )
        ]
    ),

    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        # added user-friendly features, text-based filter, sorting, and pagination
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single", # necessary for callback to function
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0], # to prevent issues
        page_action="native",
        page_current=0,
        page_size=10,
        # table size changed to fit window without horizontal scrolling
        style_table={
        'maxHeight': '500px',
        'overflowY': 'auto',
        'overflowX': 'auto'
        }
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that the chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
@app.callback( [Output('datatable-id','data'),
                Output('datatable-id','selected_rows')],
               [Input('filter-type', 'value')] )
def update_dashboard(filter_type):
    # Base query (Reset)
    query = {}

    # Apply filter based on radio selection
    # Query is based on the requirements
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesa Bay Retr", # Chesapeake Bay Retriever does not seem to be in dataset, only mix with this shortened name
                    "Chesapeake Bay Retriever", # incase it is added under this name in the future
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinsch", # This is what the Doberman Pinschers are called in the dataset
                    "Doberman Pinscher", # incase it is added under this name in the future
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    # If filter_type is 'Reset' or not present, the base query is used

    # Read from MongoDB using query
    df = pd.DataFrame.from_records(shelter.read(query))

    # Clean up MongoDB _id field
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    return df.to_dict('records'), [0] # Sends new set of records and sets selected row back to the first one

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if not viewData: # Should only happen when program first starts
        return [html.Div("Please wait for data to load!")]

    dff = pd.DataFrame(viewData)
    
    # The below is specifically to highlight the top breeds and put the rest into a single category, or else the pie chart becomes bloated
    # Count breeds
    counts = dff['breed'].value_counts()

    # Keep top breeds
    top_n = 10
    top_breeds = counts.head(top_n)

    # Group the rest into "Other". While this loses some specific data, it makes it so the pie chart is more understandable
    other_total = counts.iloc[top_n:].sum() # Puts the number of everything else together
    if other_total > 0:
        top_breeds["Other Breeds"] = other_total

    # Builds a DataFrame for Plotly
    pie_df = pd.DataFrame({
        "Breed": top_breeds.index,
        "Amount": top_breeds.values
    })

    # Builds pie chart
    fig = px.pie(
        pie_df,
        names="Breed",
        values="Amount",
        title="Breeds",
        color_discrete_sequence=px.colors.sequential.Turbo # alternate color
    )

    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Callback updates geo-location chart
# derived_virtual_data is set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index): # geolocation chart code

    dff = pd.DataFrame.from_dict(viewData)
    
    # Fixes certain errors (error at start-up)
    if not index:
        return no_update

    # Because we only allow single row selection, the list can 
    # be converted to a row index here
    if index is None or index[0] >= len(dff):
        row = 0
    else: 
        row = index[0]

    # Column 13 and 14 define the grid-coordinates for 
    # the map. Set as variables for easier use
    lat = dff.iloc[row, 13]
    lon = dff.iloc[row, 14]
    
    # Column 9 has the animal's name, column 4 has the animal's breed
    name = dff.iloc[row,9]
    breed = dff.iloc[row,4]
    
    if name is None or str(name).strip() == "": # Sometimes the animal has no name attached
        name = "Unknown Name"
    return [
    dl.Map(style={'width': '1000px', 'height': '500px'},
       center=[lat, lon], zoom=10, children=[ # Made it so it centers on the actual marker/position
       dl.TileLayer(id="base-layer-id"),
       # Marker with tool tip and popup
       dl.Marker(position=[lat,lon],
          children=[
          dl.Tooltip(breed),
          dl.Popup([
            html.P(name) ## Shows animal name when clicking
         ])
      ])
   ])
]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://eclipseneptune-augustprefix-3000.codio.io/proxy/8050/
